In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from cmdstanpy import CmdStanModel

import arviz as az

Using the 2026 dataset: https://memory.psych.upenn.edu/Data_Archive#2026

In [111]:
# Get data
data = pd.read_csv('data/Exp3_AllData.csv', header=0, sep=',')
data.head()
data = data[~np.isnan(data["Output_Rank"])]

# Convert to ints
data["Listnum"] = data["Listnum"].astype(int)
data["Spatial_Input_Order"] = data["Spatial_Input_Order"].astype(int)
data["Temporal_Input_Position"] = data["Temporal_Input_Position"].astype(int)
data["Spatial_Recall_Order"] = data["Spatial_Recall_Order"].astype(int)
data["Length"] = data["Length"].astype(int)
data["Correct"] = data["Correct"].astype(int)
data["Distance_From_Correct"] = data["Distance_From_Correct"].astype(int)
data["Serial_Pos_Encoded"] = data["Serial_Pos_Encoded"].astype(int)
data["Output_Rank"] = data["Output_Rank"].astype(int)

# Only use a fraction of the data
train_percent = 0.3
train_n = int(data.shape[0]*train_percent)
data = data.sample(n=train_n)

modif = {}
newv = []
for strin in (data["Condition"].values):
    if not(strin in modif.keys()):
        if len(modif) != 0:
            modif[strin] = modif[max(modif, key=modif.get)]+1
        else:
            modif[strin] = 1
        newv.append(modif[strin])
    else:
        newv.append(modif[strin])
    

data_dict = {
    # Sizes
    "N": data.shape[0],
    # Experiment
    "spatial_input_order": data.Spatial_Input_Order.values,
    "temporal_input_pos": data.Temporal_Input_Position.values,
    # Results
    "correct": data.Correct.values,
    #"output_rank": data.Output_Rank.values,
    "rt": data.Initial_RT_Time.values,
    "type": np.array(newv),
}

In [112]:
data.Temporal_Input_Position.values

array([ 1,  1, 11, ...,  1,  3,  7], shape=(10590,))

In [116]:
# Compile model
model = CmdStanModel(stan_file="stan/model.stan")

19:11:23 - cmdstanpy - INFO - compiling stan file /mnt/c/Users/inter/CompOrg/TinyCC/stan/model.stan to exe file /mnt/c/Users/inter/CompOrg/TinyCC/stan/model
19:11:36 - cmdstanpy - INFO - compiled model executable: /mnt/c/Users/inter/CompOrg/TinyCC/stan/model


In [118]:
# Sample model
fit = model.sample(data=data_dict, chains=4, iter_sampling=7500, iter_warmup=1000)

# Display sampling diagnostics
print(fit.diagnose())

19:14:43 - cmdstanpy - INFO - CmdStan start processing








chain 1:   0%|                                                                       | 0/8500 [00:00<?, ?it/s, (Warmup)]








chain 2:   0%|                                                                       | 0/8500 [00:00<?, ?it/s, (Warmup)]









chain 3:   0%|                                                                       | 0/8500 [00:00<?, ?it/s, (Warmup)]










chain 4:   0%|                                                                       | 0/8500 [00:00<?, ?it/s, (Warmup)]







chain 1:   0%|                                                               | 1/8500 [00:00<14:26,  9.81it/s, (Warmup)]








chain 1: 100%|██████████████████████████████████████████████████████████| 8500/8500 [03:07<00:00, 66.23it/s, (Sampling)]

chain 2:  73%|██████████████████████████████████████████▎               | 6200/8500 [03:17<00:50, 45.31it/s, (Sampling)]


chain 4:  75%|███████████████████████████████


19:20:23 - cmdstanpy - INFO - CmdStan done processing.


19:20:24 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 7367 divergent transitions (98.2%)
	Chain 2 had 7481 divergent transitions (99.7%)
	Chain 3 had 7462 divergent transitions (99.5%)
	Chain 4 had 7385 divergent transitions (98.5%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
29699 of 30000 (99.00%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  params[2,1], params[4,1]
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



In [119]:
fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-5782.250000,0.075219,3.601370,3.560460e+00,-5.788620e+03,-5.781930e+03,-5.776910e+03,2245.790,4686.800,2.548870,1.00305
"params[1,1]",-0.173472,0.005337,0.117362,1.190940e-01,-3.681850e-01,-1.710730e-01,1.586610e-02,498.344,1040.240,0.565596,1.00414
"params[1,2]",0.009271,0.000111,0.004630,4.558570e-03,1.613370e-03,9.260800e-03,1.677200e-02,1725.950,4221.460,1.958860,1.00147
"params[1,3]",0.011481,0.000534,0.014005,1.411900e-02,-1.101340e-02,1.123060e-02,3.487130e-02,709.129,1542.900,0.804827,1.00358
"params[2,1]",-1.497690,0.013203,0.172707,1.649430e-01,-1.783570e+00,-1.491700e+00,-1.217120e+00,188.043,126.600,0.213419,1.01538
"params[2,2]",0.004893,0.000274,0.007008,6.941810e-03,-6.689520e-03,4.847640e-03,1.644300e-02,649.664,1277.250,0.737337,1.00617
"params[2,3]",-0.006189,0.001068,0.020471,2.060310e-02,-3.913670e-02,-6.429110e-03,2.786790e-02,364.507,504.369,0.413698,1.00647
"params[3,1]",-1.262710,0.004956,0.166123,1.699300e-01,-1.532590e+00,-1.263770e+00,-9.881700e-01,1131.570,2122.500,1.284280,1.00411
"params[3,2]",0.003847,0.000136,0.006564,6.497010e-03,-7.091610e-03,3.956570e-03,1.451100e-02,2328.230,4023.650,2.642430,1.00230
"params[3,3]",-0.020977,0.000555,0.020260,2.059220e-02,-5.385040e-02,-2.095090e-02,1.283140e-02,1337.880,2301.770,1.518430,1.00226
